<a href="https://colab.research.google.com/github/pavloslee/pavloslee.github.io/blob/main/TMR4130_bayesian_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayesian network

Text ✅

In [1]:

# Step 1: Install pgmpy in Colab
!pip install pgmpy -q


# Step 2: Import libraries
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# Step 3: Define the network structure (Edge from Rain -> Sprinkler)
model = DiscreteBayesianNetwork([('Rain', 'Sprinkler')])

# Step 4: Define the CPT for the TOP node (Rain)
# P(Rain = False) = 0.8, P(Rain = True) = 0.2
cpd_rain = TabularCPD(
    variable='Rain',
    variable_card=2,
    values=[[0.8], [0.2]],
    state_names={'Rain': ['False', 'True']}
)

# Step 5: Define the CPT for the CHILD node (Sprinkler) given Rain
# P(Sprinkler | Rain)
cpd_sprinkler = TabularCPD(
    variable='Sprinkler',
    variable_card=2,
    values=[
        [0.6, 0.99],  # P(Sprinkler = False | Rain=False, Rain=True)
        [0.4, 0.01]   # P(Sprinkler = True  | Rain=False, Rain=True)
    ],
    evidence=['Rain'],
    evidence_card=[2],
    state_names={
        'Sprinkler': ['False', 'True'],
        'Rain': ['False', 'True']
    }
)

# Step 6: Add CPTs to model and validate network structure
model.add_cpds(cpd_rain, cpd_sprinkler)
assert model.check_model()  # Checks that probabilities sum to 1.0

# Step 7: Perform inference on the TOP node (Rain)
inference = VariableElimination(model)

# Query 1: Prior probability of the top node (Rain)
top_node_prior = inference.query(variables=['Rain'])
print("--- Prior Distribution of Top Node (Rain) ---")
print(top_node_prior)

# Query 2: Infer top node (Rain) given evidence about the bottom node (Sprinkler = True)
top_node_posterior = inference.query(
    variables=['Rain'],
    evidence={'Sprinkler': 'True'}
)
print("\n--- Posterior Distribution of Top Node (Rain | Sprinkler=True) ---")
print(top_node_posterior)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 13.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


--- Prior Distribution of Top Node (Rain) ---
+-------------+-------------+
| Rain        |   phi(Rain) |
+=============+=============+
| Rain(False) |      0.8000 |
+-------------+-------------+
| Rain(True)  |      0.2000 |
+-------------+-------------+

--- Posterior Distribution of Top Node (Rain | Sprinkler=True) ---
+-------------+-------------+
| Rain        |   phi(Rain) |
+=============+=============+
| Rain(False) |      0.9938 |
+-------------+-------------+
| Rain(True)  |      0.0062 |
+-------------+-------------+
